In [ ]:
%pip install pandas datasets torch transformers peft scikit-learn evaluate numpy

In [ ]:
from datasets import load_from_disk, load_dataset
# Load the dataset
# Save the dataset locally
# dataset = load_dataset('fancyzhx/ag_news')
dataset = load_from_disk('ag_news')

dataset = dataset.rename_column('label','labels')


# Load the dataset from local
dataset['train'] = dataset['train'].shuffle(seed=42).select(range(500))
dataset['test'] = dataset['test'].shuffle(seed=42).select(range(100))

dataset['train']

In [ ]:
import torch
print(torch.cuda.is_available())  # Ensure CUDA is available
print(torch.cuda.current_device())  # Check the current device
print(torch.__version__)  # Check PyTorch version
print(torch.cuda.get_device_name(0)) 

In [ ]:
from transformers import AutoTokenizer, BertTokenizer


tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
train_dataset = dataset['train']
test_dataset = dataset['train']

X = tokenizer(train_dataset['text'], padding=True, truncation=True, max_length=512)
X1 = tokenizer(test_dataset['text'], padding=True, truncation=True, max_length=512)
y = train_dataset['labels']
y1 = test_dataset['labels']

In [ ]:
import torch
# Create torch dataset
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

In [ ]:
train_dataset = Dataset(X, y)
test_dataset = Dataset(X1, y1)

In [ ]:
from transformers import TrainingArguments


# Define the training arguments
training_args = TrainingArguments(
    output_dir="./results1",            # Where to save the model checkpoints
    eval_strategy='steps',
    # learning_rate=2e-5,                # Learning rate for training
    per_device_train_batch_size=8,    # Batch size for training
    per_device_eval_batch_size=8,     # Batch size for evaluation
    num_train_epochs=1,                # Number of epochs
    # weight_decay=0.01,                 # Weight decay for regularization
    logging_dir="./logs",              # Where to save logs
    logging_steps=10,                  # Log every 10 steps
    label_names=['labels'],
    # load_best_model_at_end=True,       # Load the best model at the end of training
    metric_for_best_model="accuracy"   # Recommended metric for classification tasks
)


In [ ]:


from peft import get_peft_model, LoraConfig
from transformers import BertForSequenceClassification
# Load the BERT base uncased model
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)
# lora_config = LoraConfig(
#     r=4,  # Low-rank size, you can experiment with this value
#     lora_alpha=32,  # Scaling factor, controlling the impact of LoRa
#     lora_dropout=0.1,  # Dropout rate for LoRa layers
# )
# # Apply LoRa configuration using get_peft_model
# model = get_peft_model(base_model, lora_config)

In [ ]:
from transformers import Trainer
from sklearn.metrics import accuracy_score
from evaluate import load
import numpy as np

accuracy = load("accuracy")

# Load metric
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    # return {'accuracy': acc, 'eval_accuracy': acc}
    return accuracy.compute(predictions=predictions, references=labels)
# Prepare the trainer
trainer = Trainer(
    model=model,   
    args=training_args,                    # Training arguments defined earlier
    train_dataset=train_dataset,   # Training dataset
    eval_dataset=test_dataset,     # Evaluation dataset
    tokenizer=tokenizer,   
    compute_metrics=compute_metrics  # Accuracy metric

)

In [ ]:
import os, torch
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
def predict_label(text: str, model, tokenizer):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        predicted_class_id = torch.argmax(logits, dim=-1).item()
    
    # print("Predicted Label:", predicted_class_id)
    return predicted_class_id
    

In [ ]:

test = dataset['test'].shuffle(seed=12).select(range(40))
cnt_matched = 0
for i in test:
    ans = predict_label(i['text'], trainer.model, trainer.tokenizer)
    if ans == i['labels']:
        cnt_matched += 1
        print("Matched the label")
        print(i['text'])
        print("Dataset Label:" + str(i['labels']))
        print("Predicted Label: " + str(ans))

print(f"Matched Results: {cnt_matched} / 40")